# Translater avec une matrice

| Module | Durée | Prérequis |
|---|---|---|
| M3 – Matrices et transformations | 75 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Inès programme un petit jeu vidéo en 2D. Dans le décor, un moulin a quatre ailes qui doivent **tourner autour du moyeu**, situé au point $(4\,;\,3)$. Le joueur peut aussi **déplacer** le moulin dans le décor.

Inès a déjà écrit des matrices de rotation, mais quand elle les applique, les ailes s'envolent à travers l'écran au lieu de tourner sur place.

**Les questions d'Inès : pourquoi ses ailes s'envolent-elles ? Comment faire tourner un objet autour d'un point qui n'est pas l'origine, et comment le déplacer, avec des matrices ?**

**Activité** : D'après le notebook 3.1, autour de quel point une matrice de rotation fait-elle tourner les objets ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_translation_2x2 = True   # existe-t-il une matrice 2 x 2 qui déplace tout le plan de (1 ; 0) ? (modifiez)

## ③ Abstraire

### Une translation n'est pas linéaire

Déplacer de $\mathbf{t} = (t_x\,;\,t_y)$, c'est la transformation $T(\mathbf{x}) = \mathbf{x} + \mathbf{t}$. Or toute matrice envoie l'origine sur elle-même : $A\mathbf{0} = \mathbf{0}$. Comme $T(\mathbf{0}) = \mathbf{t} \neq \mathbf{0}$, **aucune matrice $2 \times 2$ ne réalise une translation**. Le concept de matrice, tel quel, ne s'applique pas.

### L'astuce : une dimension de plus

On ajoute à chaque point une troisième coordonnée, toujours égale à 1 : le point $(x\,;\,y)$ devient $(x\,;\,y\,;\,1)$. Ce sont les **coordonnées homogènes**. Alors la translation devient un produit matriciel :

$$
\begin{pmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{pmatrix}
\begin{pmatrix} x \\ y \\ 1 \end{pmatrix}
=
\begin{pmatrix} x + t_x \\ y + t_y \\ 1 \end{pmatrix}
$$

![Schéma : le plan du dessin placé à la hauteur z = 1 dans l'espace ; une translation dans ce plan est un cisaillement de l'espace](../../assets/schemas/schema-homogenes.svg)

Les transformations du notebook 3.1 se recopient dans le coin supérieur gauche d'une matrice $3 \times 3$ :

$$
A_h = \begin{pmatrix} a & b & 0 \\ c & d & 0 \\ 0 & 0 & 1 \end{pmatrix}
$$

### Tourner autour d'un point quelconque

Pour tourner autour du point $\mathbf{c}$, on combine trois étapes : amener $\mathbf{c}$ à l'origine, tourner, puis ramener. Avec la règle du notebook 3.2 (le produit se lit de droite à gauche) :

$$
\text{Rotation autour de } \mathbf{c} \;=\; T_{\mathbf{c}} \, R_\theta \, T_{-\mathbf{c}}
$$

**Activité** : Calculez le produit $T_{\mathbf{t}}\,R_{90^\circ}$ pour $\mathbf{t} = (2\,;\,0)$, puis $R_{90^\circ}\,T_{\mathbf{t}}$. Où chacun envoie-t-il l'origine ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Un point est une COLONNE : un ensemble de n points est un tableau de forme (2, n).
# Ainsi, appliquer la matrice A à tous les points s'écrit simplement A @ points.

# La lettre F : asymétrique, elle montre les rotations ET les retournements
F = np.array([[0, 0, 2, 2, 0.5, 0.5, 1.5, 1.5, 0.5, 0.5, 0],
              [0, 3, 3, 2.5, 2.5, 1.7, 1.7, 1.2, 1.2, 0, 0]])

def grille(ax, A=np.eye(2), n=4, **style):
    # Trace l'image par A des droites x = k et y = k, pour k entre -n et n
    t = np.linspace(-n, n, 2)
    for k in range(-n, n + 1):
        for ligne in (np.array([t, np.full(2, k)]), np.array([np.full(2, k), t])):
            image = A @ ligne
            ax.plot(*image, lw=0.6, color=style.get("couleur", "#D5DBDE"), zorder=0)

def dessiner(ax, A, titre="", limite=4.5):
    # Grille transformée, lettre F avant (gris) et après (vert), images des vecteurs de base
    grille(ax, A)
    ax.fill(*F, color=LIOUAI["gris"], alpha=0.35, lw=0)
    image = A @ F
    ax.fill(*image, color=LIOUAI["vert"], alpha=0.55)
    ax.plot(*image, color=LIOUAI["vert_texte"], lw=1.5)
    for colonne, couleur in [(A[:, 0], LIOUAI["vert_texte"]), (A[:, 1], LIOUAI["navy"])]:
        ax.annotate("", xy=colonne, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", lw=2, color=couleur, mutation_scale=14))
    ax.set_xlim(-limite, limite); ax.set_ylim(-limite, limite); ax.set_aspect("equal")
    ax.axhline(0, color=LIOUAI["gris"], lw=0.8); ax.axvline(0, color=LIOUAI["gris"], lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titre, fontsize=11)

# Le catalogue des transformations
def rotation(theta_deg):
    t = np.radians(theta_deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def homothetie(k):
    return k * np.eye(2)

def dilatation(kx, ky):
    return np.array([[kx, 0], [0, ky]])

def cisaillement(k):
    return np.array([[1, k], [0, 1]])

def projection_sur_x():
    return np.array([[1, 0], [0, 0]])

def symetrie_axe_y():
    return np.array([[-1, 0], [0, 1]])

In [ ]:
def en_homogene(points):
    # (2, n) -> (3, n) : ajoute une ligne de 1
    return np.vstack([points, np.ones(points.shape[1])])

def depuis_homogene(points_h):
    return points_h[:2]

def translation_h(tx, ty):
    return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=float)

def lineaire_h(A):
    # Recopie une matrice 2 x 2 dans une matrice homogène 3 x 3
    A_h = np.eye(3)
    A_h[:2, :2] = A
    return A_h

def rotation_h(theta_deg):
    return lineaire_h(rotation(theta_deg))

print(translation_h(1, 0) @ np.array([0, 0, 1]), "<- l'origine est bien déplacée")

### Le moulin d'Inès

In [ ]:
moyeu = np.array([4.0, 3.0])

# Une aile : un quadrilatère allongé, dessiné pointe vers la droite, à partir du moyeu
aile = np.array([[0, 0.4, 2.6, 2.4, 0], [0, -0.25, -0.15, 0.35, 0]]) + moyeu[:, None]
# Le mât : un trapèze sous le moyeu
mat = np.array([[-0.6, 0.6, 0.35, -0.35], [-3, -3, 0, 0]]) + moyeu[:, None]

def rotation_h_autour(theta, centre):
    # Amener le centre à l'origine, tourner, ramener : le produit se lit de droite à gauche
    return translation_h(*centre) @ rotation_h(theta) @ translation_h(*-np.asarray(centre))

def ailes_du_moulin(angle, M=np.eye(3)):
    # Les 4 ailes, tournées de « angle » autour du moyeu, puis transformées par M
    return [depuis_homogene(M @ rotation_h_autour(angle + 90 * k, moyeu) @ en_homogene(aile)) for k in range(4)]

def dessiner_moulin(ax, liste_ailes, titre, M=np.eye(3)):
    ax.fill(*depuis_homogene(M @ en_homogene(mat)), color=LIOUAI["gris"], alpha=0.6)
    for p in liste_ailes:
        ax.fill(*p, color=LIOUAI["vert"], alpha=0.7); ax.plot(*p, color=LIOUAI["vert_texte"], lw=1)
    ax.plot(*depuis_homogene(M @ np.array([*moyeu, 1])), "o", color=LIOUAI["navy"])
    ax.set_xlim(-4, 12); ax.set_ylim(-4, 9); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.axhline(0, color=LIOUAI["gris"], lw=0.8); ax.axvline(0, color=LIOUAI["gris"], lw=0.8)
    ax.set_title(titre, fontsize=10)

# Le bug d'Inès : elle tourne autour de l'origine
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ailes_bug = [depuis_homogene(rotation_h(40) @ rotation_h_autour(k * 90, moyeu) @ en_homogene(aile)) for k in range(4)]
dessiner_moulin(axes[0], ailes_bug, "Bug : rotation de 40° autour de l'origine")
dessiner_moulin(axes[1], ailes_du_moulin(40), "Correct : rotation de 40° autour du moyeu")
plt.tight_layout(); plt.show()

**Activité** : Sur le graphique de gauche, où sont parties les ailes ? Expliquez le bug d'Inès en une phrase.

Indiquez vos réponses et justifiez-les.

### L'animation et le déplacement

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, angle in zip(axes, [0, 20, 40, 60]):
    dessiner_moulin(ax, ailes_du_moulin(angle), f"angle {angle}°")
plt.suptitle("Les ailes tournent sur place"); plt.tight_layout(); plt.show()

# Le joueur déplace le moulin de (5 ; -1) : on compose la translation APRÈS la rotation
deplacement = translation_h(5, -1)
fig, ax = plt.subplots(figsize=(6, 4.8))
dessiner_moulin(ax, ailes_du_moulin(25, M=deplacement), "Moulin déplacé de (5 ; -1), ailes à 25°", M=deplacement)
plt.show()

## ⑤ Interpréter

In [ ]:
print("Votre pari « translation par une 2 x 2 » :", pari_translation_2x2, "; réalité : impossible, car A @ 0 = 0")
T, R90 = translation_h(2, 0), rotation_h(90)
print("T R : l'origine va en", depuis_homogene(T @ R90 @ np.array([0, 0, 1])))
print("R T : l'origine va en", depuis_homogene(R90 @ T @ np.array([0, 0, 1])))
print("Dernière ligne de T R :", (T @ R90)[2])

**Activité** :

1. « Tourner puis déplacer » et « déplacer puis tourner » donnent-ils le même résultat ? Reliez votre réponse au notebook 3.2.
2. Pourquoi la dernière ligne de toutes ces matrices reste-t-elle $(0\;\;0\;\;1)$ ? Que se passerait-il si elle changeait ?
3. Dans un vrai jeu, le moulin, le personnage et la caméra ont chacun leur transformation. Comment calculer en une seule matrice la position d'une aile à l'écran ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le coût de l'astuce.** Une matrice homogène a 9 coefficients, dont 3 sont toujours $0$, $0$ et $1$. Pourquoi les moteurs de jeu l'utilisent-ils malgré tout ?
- **Vérifier autrement.** Après rotation autour du moyeu, le moyeu doit rester immobile et chaque aile doit garder sa longueur. Vérifiez-le par le calcul.
- **Généraliser.** Écrivez une fonction `homothetie_autour(k, centre)` qui agrandit un objet par $k$ **sans déplacer** le point `centre`. Testez-la en agrandissant le moulin × 1,5 autour de la base de son mât.

In [ ]:
def homothetie_autour(k, centre):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def homothetie_autour(k, centre):
    return translation_h(*centre) @ lineaire_h(homothetie(k)) @ translation_h(*-np.asarray(centre))

rot = rotation_h_autour(73, moyeu)
print("Moyeu après rotation :", depuis_homogene(rot @ np.array([*moyeu, 1])))
longueur = lambda p: np.linalg.norm(p[:, 2] - p[:, 0])
print("Longueur d'une aile avant / après :", round(longueur(aile), 6), round(longueur(depuis_homogene(rot @ en_homogene(aile))), 6))

base = moyeu + np.array([0, -3])
H = homothetie_autour(1.5, base)
fig, ax = plt.subplots(figsize=(6, 4.8))
dessiner_moulin(ax, ailes_du_moulin(15, M=H), "Moulin × 1,5 autour de la base du mât", M=H)
ax.plot(*base, "s", color=LIOUAI["jaune"], ms=9)
plt.show()

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Proposer des abstractions alternatives": 1,
    "Repérer ressemblances et différences entre situations pour abstraire": 1,
    "Reconnaître si le concept s'applique": 1,
    "Combiner des outils pour produire le résultat": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "3.3-coordonnees-homogenes", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")